Phase 4 — Master Dataset Construction

This notebook builds the collision-level Master Dataset by joining the validated Vehicle and Casualty aggregation tables (from Phase 3) onto the cleaned Collision base table (from Phase 2). Tasks 10-13: prepare the base table, left-join both aggregates, and verify integrity — one row per collision, no unexpected row multiplication, and count consistency preserved throughout.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")

collision_df = pd.read_csv(data_path / "collision_cleaned.csv", dtype={"collision_index": str})
vehicle_agg = pd.read_csv(data_path / "vehicle_agg.csv", dtype={"collision_index": str})
casualty_agg = pd.read_csv(data_path / "casualty_agg.csv", dtype={"collision_index": str})

print("Collision:", collision_df.shape)
print("Vehicle aggregation:", vehicle_agg.shape)
print("Casualty aggregation:", casualty_agg.shape)

C:\Users\ACER\AppData\Local\Temp\ipykernel_22492\2802769786.py:7: DtypeWarning: Columns (2: collision_ref_no) have mixed types. Specify dtype option on import or set low_memory=False.
  collision_df = pd.read_csv(data_path / "collision_cleaned.csv", dtype={"collision_index": str})


Collision: (513801, 44)
Vehicle aggregation: (513801, 13)
Casualty aggregation: (513801, 6)


Task 10 — Prepare COLLISION as the Base Table

Before joining, confirm the cleaned Collision table's row count, collision_index uniqueness, and target-field availability, establishing it as the fixed base population for the Master Dataset.

In [5]:
print("Collision base row count:", len(collision_df))
print("Unique collision_index in Collision:", collision_df["collision_index"].nunique())
print("collision_index uniqueness check (should match row count):",
      len(collision_df) == collision_df["collision_index"].nunique())

print("\nTarget field availability:")
print("collision_severity missing:", collision_df["collision_severity"].isna().sum())
print("number_of_casualties missing:", collision_df["number_of_casualties"].isna().sum())

print("\ncollision_severity distribution:")
print(collision_df["collision_severity"].value_counts().sort_index())

Collision base row count: 513801
Unique collision_index in Collision: 513801
collision_index uniqueness check (should match row count): True

Target field availability:
collision_severity missing: 0
number_of_casualties missing: 0

collision_severity distribution:
collision_severity
1      7553
2    116813
3    389435
Name: count, dtype: int64


Task 11 — LEFT JOIN Aggregated VEHICLE

Attach the validated Vehicle aggregation table to the Collision base using a left join on collision_index, ensuring no base collisions are lost and no rows are unexpectedly duplicated. Since Task 8.7 already confirmed a 100% count match between Vehicle and Collision, this join is expected to introduce zero missing aggregates and zero row multiplication.

In [6]:
row_count_before = len(collision_df)

master_df = collision_df.merge(vehicle_agg, on="collision_index", how="left")

print("Row count before join:", row_count_before)
print("Row count after Vehicle join:", len(master_df))
print("Row count unchanged:", row_count_before == len(master_df))

print("\ncollision_index uniqueness after join:", master_df["collision_index"].is_unique)

print("\nMissing Vehicle aggregate rows (should be 0, given 100% match in Task 8.7):")
vehicle_agg_cols = [col for col in vehicle_agg.columns if col != "collision_index"]
print(master_df[vehicle_agg_cols].isna().all(axis=1).sum())

print("\nSample of joined data:")
master_df[["collision_index", "number_of_vehicles"] + vehicle_agg_cols[:4]].head()

Row count before join: 513801
Row count after Vehicle join: 513801
Row count unchanged: True

collision_index uniqueness after join: True

Missing Vehicle aggregate rows (should be 0, given 100% match in Task 8.7):
0

Sample of joined data:


,collision_index,number_of_vehicles,has_bus_or_coach,has_car,has_hgv,has_motorcycle
0,202417H105824,2,0,1,0,0
1,202517M115025,2,0,1,0,0
2,202517L306225,2,0,1,0,0
3,202417S104724,2,0,1,0,0
4,202517S106525,2,0,1,0,0


Task 12 — LEFT JOIN Aggregated CASUALTY

Attach the validated Casualty aggregation table to master_df using a left join on collision_index, ensuring no rows are lost or duplicated. Since Task 9.4 already confirmed a 100% count match between Casualty and Collision, this join is expected to introduce zero missing aggregates and zero row multiplication — completing the integrated Master Dataset candidate.

In [7]:
row_count_before = len(master_df)

master_df = master_df.merge(casualty_agg, on="collision_index", how="left")

print("Row count before Casualty join:", row_count_before)
print("Row count after Casualty join:", len(master_df))
print("Row count unchanged:", row_count_before == len(master_df))

print("\ncollision_index uniqueness after join:", master_df["collision_index"].is_unique)

print("\nMissing Casualty aggregate rows (should be 0, given 100% match in Task 9.4):")
casualty_agg_cols = [col for col in casualty_agg.columns if col != "collision_index"]
print(master_df[casualty_agg_cols].isna().all(axis=1).sum())

print("\nFinal Master Dataset shape:", master_df.shape)
print("\nSample of fully joined data:")
master_df[["collision_index", "number_of_casualties", "collision_severity"] + casualty_agg_cols].head()

Row count before Casualty join: 513801
Row count after Casualty join: 513801
Row count unchanged: True

collision_index uniqueness after join: True

Missing Casualty aggregate rows (should be 0, given 100% match in Task 9.4):
0

Final Master Dataset shape: (513801, 61)

Sample of fully joined data:


,collision_index,number_of_casualties,collision_severity,pedestrian_involved,has_passenger_casualty,has_child_casualty,has_elderly_casualty,worst_casualty_severity
0,202417H105824,1,2,False,False,False,False,2
1,202517M115025,1,2,False,False,False,False,2
2,202517L306225,3,3,False,True,True,False,3
3,202417S104724,2,3,False,True,False,False,3
4,202517S106525,1,2,False,False,False,False,2


Task 13 — Verify Master Dataset Integrity (Mandatory Gate)

Perform a final comprehensive validation of the integrated Master Dataset before proceeding to feature engineering: total row count against the Collision base population, collision_index uniqueness, unchanged number_of_vehicles/number_of_casualties values, plausible aggregate ranges, and no unexpected null expansion. This is the last checkpoint before the dataset is considered validated.

In [8]:
print("=" * 70)
print("TASK 13 — MASTER DATASET INTEGRITY VERIFICATION")
print("=" * 70)

# 1. Row count against Collision base population
print("\n1. Row count check:")
print(f"   Master Dataset rows: {len(master_df)}")
print(f"   Collision base rows: {len(collision_df)}")
print(f"   Match: {len(master_df) == len(collision_df)}")

# 2. collision_index uniqueness
print("\n2. collision_index uniqueness:")
print(f"   Unique: {master_df['collision_index'].is_unique}")
print(f"   Duplicates: {master_df['collision_index'].duplicated().sum()}")

# 3. number_of_vehicles / number_of_casualties unchanged from Collision
print("\n3. Target/count field integrity:")
print(f"   number_of_vehicles unchanged: {(master_df['number_of_vehicles'] == collision_df['number_of_vehicles']).all()}")
print(f"   number_of_casualties unchanged: {(master_df['number_of_casualties'] == collision_df['number_of_casualties']).all()}")
print(f"   collision_severity missing: {master_df['collision_severity'].isna().sum()}")

# 4. Plausible aggregate ranges
print("\n4. Aggregate range checks:")
print(f"   vehicle_type_diversity range: {master_df['vehicle_type_diversity'].min()} to {master_df['vehicle_type_diversity'].max()}")
print(f"   min_driver_age range: {master_df['min_driver_age'].min()} to {master_df['min_driver_age'].max()}")
print(f"   worst_casualty_severity range: {master_df['worst_casualty_severity'].min()} to {master_df['worst_casualty_severity'].max()}")

# 5. No unexpected null expansion — compare missingness before/after join
print("\n5. Null expansion check on original Collision columns:")
original_missing = collision_df.isna().sum().sum()
master_original_cols_missing = master_df[collision_df.columns].isna().sum().sum()
print(f"   Missing values in Collision columns before join: {original_missing}")
print(f"   Missing values in same columns within Master Dataset: {master_original_cols_missing}")
print(f"   Unchanged: {original_missing == master_original_cols_missing}")

print("\n" + "=" * 70)
print(f"FINAL MASTER DATASET SHAPE: {master_df.shape}")
print("=" * 70)

TASK 13 — MASTER DATASET INTEGRITY VERIFICATION

1. Row count check:
   Master Dataset rows: 513801
   Collision base rows: 513801
   Match: True

2. collision_index uniqueness:
   Unique: True
   Duplicates: 0

3. Target/count field integrity:
   number_of_vehicles unchanged: True
   number_of_casualties unchanged: True
   collision_severity missing: 0

4. Aggregate range checks:
   vehicle_type_diversity range: 1 to 6
   min_driver_age range: 1.0 to 100.0
   worst_casualty_severity range: 1 to 3

5. Null expansion check on original Collision columns:
   Missing values in Collision columns before join: 215
   Missing values in same columns within Master Dataset: 215
   Unchanged: True

FINAL MASTER DATASET SHAPE: (513801, 61)


Task 13 — Complete. Master Dataset Validated.

All integrity checks passed: row count preserved (513,801), collision_index unique, number_of_vehicles/number_of_casualties unchanged from source, plausible aggregate ranges, and zero unexpected null expansion in original Collision columns. This is the validated one-row-per-collision Master Dataset, ready for Phase 5 (Feature Engineering).

In [9]:
output_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")

master_df.to_csv(output_path / "master_dataset_v1.csv", index=False)

print("Master Dataset saved to:", output_path.resolve())
print("Shape:", master_df.shape)

Master Dataset saved to: C:\Users\ACER\Downloads\New folder\cleaned_datasets
Shape: (513801, 61)
